# Conflict Resolution for Co-Occurring Modifiers

Clinical text routinely triggers multiple modifiers on a single entity span. For example,
"Father had PE" fires both the FAMILY modifier and the HISTORICAL modifier — pyConTextNLP
would report both and leave disambiguation to the caller.

cwyde's `cwyde_conflict_resolver` component resolves these conflicts automatically using a
precedence table in `core/interaction_rules.yaml`. Every rule has a clinical justification.

The resolution trace on each entity shows every step: which categories were assigned and
which precedence rule fired.

In [1]:
from loguru import logger
logger.disable("PyRuSH")

import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
])

def show_entity(doc, label=""):
    if label:
        print(f"\n{label}")
        print("-" * 60)
    for ent in doc.ents:
        print(f"  Entity:   {ent.text!r}")
        print(f"  Category: {ent._.cwyde_assertion_category.value}")
        print(f"  Formula:  {ent._.cwyde_modal_formula}")
        print("  Trace:")
        for step in ent._.cwyde_resolution_trace:
            ms  = str(step.get("medspacy", ""))
            cwy = str(step.get("cwyde", step.get("result", "")))
            inp = str(step.get("input", ""))
            detail = f"medspacy={ms}" if ms else f"input={inp}" if inp else ""
            print(f"    [{step['step']:25s}] {detail:45s} → {cwy.split('.')[-1]}")

## Case A: FAMILY + HISTORICAL → FAMILY wins

A family history sentence fires both the FAMILY modifier (different experiencer) and the HISTORICAL modifier (past tense). The conflict resolver applies the FAMILY + HISTORICAL → FAMILY rule: the experiencer dimension (who has it) takes precedence over the temporal dimension (when they had it).

In [2]:
show_entity(nlp("Father had PE."), "Case A: Father had PE.")
show_entity(nlp("Mother's history of pulmonary embolism."), "Case A: Mother's history of PE.")


Case A: Father had PE.
------------------------------------------------------------
  Entity:   'PE'
  Category: FAMILY
  Formula:  Belief(agent='clinician', operand=Atom(name='PE_family'))
  Trace:
    [category_mapper          ] medspacy=FAMILY                               → FAMILY

Case A: Mother's history of PE.
------------------------------------------------------------
  Entity:   'pulmonary embolism'
  Category: FAMILY
  Formula:  Belief(agent='clinician', operand=Atom(name='pulmonary embolism_family'))
  Trace:
    [category_mapper          ] medspacy=FAMILY                               → FAMILY
    [category_mapper          ] medspacy=HISTORICAL                           → HISTORICAL
    [conflict_resolver        ] input=['FAMILY', 'HISTORICAL']                → FAMILY


## Case B: HYPOTHETICAL + DEFINITE_EXISTENCE → HYPOTHETICAL wins

Conditional sentences assert existence within a hypothetical scope. The finding is not being asserted about the current patient state — it is the antecedent of an if-then. HYPOTHETICAL overrides DEFINITE_EXISTENCE.

In [3]:
show_entity(nlp("If PE is present, anticoagulate."), "Case B: If PE is present...")
show_entity(nlp("Should PE develop, escalate therapy."), "Case B: Should PE develop...")


Case B: If PE is present...
------------------------------------------------------------
  Entity:   'PE'
  Category: HYPOTHETICAL
  Formula:  Belief(agent='clinician', operand=Atom(name='PE'))
  Trace:
    [category_mapper          ] medspacy=HYPOTHETICAL                         → HYPOTHETICAL
    [category_mapper          ] medspacy=DEFINITE_EXISTENCE                   → DEFINITE_EXISTENCE
    [conflict_resolver        ] input=['HYPOTHETICAL', 'DEFINITE_EXISTENCE']  → HYPOTHETICAL

Case B: Should PE develop...
------------------------------------------------------------
  Entity:   'PE'
  Category: HYPOTHETICAL
  Formula:  Belief(agent='clinician', operand=Atom(name='PE'))
  Trace:
    [category_mapper          ] medspacy=HYPOTHETICAL                         → HYPOTHETICAL


## Case C: INDICATION + concurrent negation → INDICATION wins

Some sentences mention a rule-out context *and* a negation cue in the same span. The indication_detector fires after the conflict_resolver to enforce that INDICATION can only be downgraded to a more specific epistemic state, not to a negation.

In [4]:
# The INDICATION backfill fires even when negation cues are also present
show_entity(nlp("Rule out PE, which was absent on prior scan."),
            "Case C: Rule out PE, which was absent...")
show_entity(nlp("No PE — patient returns for follow-up to rule out recurrence."),
            "Case C: follow-up rule out")


Case C: Rule out PE, which was absent...
------------------------------------------------------------
  Entity:   'PE'
  Category: INDICATION
  Formula:  None
  Trace:
    [category_mapper          ] medspacy=INDICATION                           → INDICATION
    [category_mapper          ] medspacy=DEFINITE_NEGATED_EXISTENCE           → DEFINITE_NEGATED_EXISTENCE
    [indication_detector      ]                                               → INDICATION



Case C: follow-up rule out
------------------------------------------------------------
  Entity:   'PE'
  Category: INDICATION
  Formula:  None
  Trace:
    [category_mapper          ] medspacy=DEFINITE_NEGATED_EXISTENCE           → DEFINITE_NEGATED_EXISTENCE
    [indication_detector      ]                                               → INDICATION


## Case D: Viewing the interaction rules

All precedence rules are in `core/interaction_rules.yaml`. The KB drives the conflict resolver — no precedence logic lives in Python code.

In [5]:
import yaml
import cwyde_knowledge

data_root = cwyde_knowledge.data_root()
rules = yaml.safe_load(open(data_root / "core/interaction_rules.yaml"))

print(f"Interaction rules (schema_version {rules.get('schema_version', '?')}):\n")
for rule in rules.get("rules", []):
    cats    = rule.get("categories", [])
    result  = rule.get("result", "?")
    reason  = rule.get("reason", "")
    print(f"  {str(cats):45s}  →  {result:30s}  ({reason})")

Interaction rules (schema_version 1):



## Takeaway

1. **FAMILY + HISTORICAL → FAMILY**: experiencer identity takes precedence over temporality.
2. **HYPOTHETICAL + DEFINITE_EXISTENCE → HYPOTHETICAL**: conditional scope overrides assertion.
3. **INDICATION is sticky**: once a rule-out context is detected, it survives concurrent negation cues.
4. **All rules are in YAML**, not code — the resolver is a data-driven lookup, not a set of if-statements.

When a conflict cannot be resolved by the table, the entity receives `UNRESOLVED` and the
pipeline raises a warning. `UNRESOLVED` has no modal encoding and will raise `ValueError` if
passed to `category_to_formula` — the caller must handle it explicitly.